---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI and Data-Driven Marketplaces

### 📋 **Topic**: Randomized Assignment and Balance Tests

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---


> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/ma/blob/main/BEFORE_YOU_BEGIN.md)

## Overview

Let's use our Python knowledge to perform a randomized assignment, and verify we did it correctly.

**What we'll learn:**
- How to perform randomized assignment
- How to check if randomization worked
- Balance tests and their importance
- Setting random seeds for reproducibility


In [ ]:
# Let's import the libraries we will use
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Load user data for randomization
df_users = pd.read_csv("../data/users.csv")

print("Dataset loaded successfully!")
print(f"Dataset shape: {df_users.shape}")
print(f"Columns: {df_users.columns.tolist()}")

print("Sample of user data:")
print(df_users.head())


## Exact Randomized Assignment

Sometimes we want exactly half the users in each group. We can achieve this using sampling:


In [ ]:
np.random.rand()*50

In [ ]:
np.random.seed(44)
df_users["treatment"] = np.where(np.random.rand(len(df_users)) < 0.2, "Treatment", "Control")

df_assignment = (
    df_users.groupby("treatment")["user"].count().reset_index(name="n")
)
print(df_assignment)

In [ ]:
df_users

## Balance Tests

If randomized assignment was performed correctly, then the treatment groups should be similar with respect to the attributes we can observe. Let's test this:


In [ ]:
# -----------------------------------------------
# Gender balance test
# -----------------------------------------------

# 1. Group data by treatment status (Treatment vs Control)
#    and gender, then count how many users are in each group.
#    `.size()` is a simple way to count rows.
df_balance_gender = (
    df_users.groupby(["treatment", "gender"])
    .size()                         # counts number of rows per (treatment, gender)
    .unstack(fill_value=0)          # turns 'gender' values into separate columns
)

# 2. Add a "Total" column to show total users in each treatment group
df_balance_gender["Total"] = df_balance_gender.sum(axis=1)

# 3. Calculate percentages within each treatment group
#    (divide each count by the row total, multiply by 100)
df_balance_gender_pct = (
    df_balance_gender.div(df_balance_gender["Total"], axis=0) * 100
).round(3)  # round to one decimal place

# 4. Print the results
print("Gender balance (counts):")
print(df_balance_gender)

print("\nGender balance (% within treatment):")
print(df_balance_gender_pct)


In [ ]:
# -----------------------------------------------
# Earnings balance test
# -----------------------------------------------

# 1. Group the data by treatment status (Treatment vs Control)
#    and calculate basic summary numbers for earnings:
#    - average earnings (mean)
#    - variation in earnings (standard deviation)
#    - how many users are in each group (count)
df_balance_earnings = (
    df_users.groupby("treatment")
    .agg({"earnings": ["mean", "std", "count"]})  # aggregate multiple stats at once
    .round(2)                                     # round numbers to 2 decimal places
)

# 2. Flatten the multi-level column names that result from using .agg()
#    so instead of ("earnings", "mean") we just have "avg_earnings", etc.
df_balance_earnings.columns = ["avg_earnings", "std_earnings", "count"]

# 3. Move "treatment" from the index back into a normal column
df_balance_earnings = df_balance_earnings.reset_index()

# 4. Print the summary table
print("Earnings balance test:")
print(df_balance_earnings)


In [ ]:
# -----------------------------------------------
# Age balance visualization
# -----------------------------------------------

# 1. Filter out users with missing or invalid ages (e.g., 0 or NaN)
#    This ensures we only plot meaningful age values.
df_users_age = df_users[df_users["age"] > 0].copy()

# 1. Create a new figure and set its size (in inches)
plt.figure(figsize=(10, 6))

# 2. Draw overlapping histograms for each treatment group
#    - data: our user dataset filtered for valid ages
#    - x="age": the variable we're plotting
#    - hue="treatment": separate colors for Treatment vs Control
#    - alpha=0.6: transparency so both histograms are visible
#    - bins=30: number of histogram bars
#    - kde=True: adds a smooth curve showing the density
sns.histplot(
    data=df_users_age,
    x="age",
    hue="treatment",
    bins=30,
    alpha=0.6,
    kde=True
)

# 3. Add axis labels and a descriptive title
plt.xlabel("Age", fontsize=12)
plt.ylabel("Number of Users", fontsize=12)
plt.title("Age Distribution by Treatment Group", fontsize=14)

# 4. Add legend and make it easier to read
plt.legend(title="Group", loc="upper right")

# 5. Add a light grid for readability
plt.grid(True, alpha=0.3)

# 6. Adjust spacing so labels and titles fit nicely
plt.tight_layout()

# 7. Save the figure to a PDF file (high resolution)
plt.savefig("../temp/age_balance_test.pdf", dpi=300, bbox_inches="tight")

# (Optional) Close the plot to free up memory in batch scripts
# plt.close()

print("✅ Age balance plot saved to '../temp/age_balance_test.pdf'")
